### Anomaly Detection and Maintenance Alerts

The purpose of this analysis is to identify unusual measurements in the robot's operating data. Unusual measurements may indicate that the robot is behaving differently from its normal operating pattern and may require further inspection.

This analysis uses the robot dataset to examine available axis measurements, identify potential anomalies, and generate potential maintenance notifications.


In [1]:
import sys
import pandas as pd

sys.path.append("../..")

from src.data_service.datacollection import StreamingSimulator

ss = StreamingSimulator("../../data/RMBR4-2_export_test.csv")

analysis_data = ss.data.copy()

print("Analysis data loaded successfully.")
print("Number of rows:", len(analysis_data))
print("Number of columns:", len(analysis_data.columns))

Analysis data loaded successfully.
Number of rows: 39672
Number of columns: 16


### Test 1: Check Available Axis Measurements

Before analyzing the robot's behaviour, we first check which axis columns contain actual measurements.

Some axis columns may contain missing values. Columns without useful measurements should not be used for anomaly detection.

This test shows the number of missing and available measurements for each axis.
 
This will tell us which of the 14 Axis columns actually contain data.

In [2]:
axis_columns = [
    column for column in analysis_data.columns
    if column.startswith("Axis")
]

print("Missing values in each Axis:")
print(analysis_data[axis_columns].isna().sum())

print("\nNumber of actual values in each Axis:")
print(analysis_data[axis_columns].notna().sum())

Missing values in each Axis:
Axis #1         0
Axis #2         0
Axis #3         0
Axis #4         0
Axis #5         0
Axis #6         0
Axis #7         0
Axis #8         0
Axis #9     39672
Axis #10    39672
Axis #11    39672
Axis #12    39672
Axis #13    39672
Axis #14    39672
dtype: int64

Number of actual values in each Axis:
Axis #1     39672
Axis #2     39672
Axis #3     39672
Axis #4     39672
Axis #5     39672
Axis #6     39672
Axis #7     39672
Axis #8     39672
Axis #9         0
Axis #10        0
Axis #11        0
Axis #12        0
Axis #13        0
Axis #14        0
dtype: int64


### Test 2: Check Zero and Non-Zero Measurements

Next, we check how often each usable robot axis has a value of zero or a non-zero value.

A zero measurement is not automatically considered a problem because the dataset contains many zero values. Therefore, we first examine the distribution of zero and non-zero measurements before identifying unusual values.


In [3]:
usable_axes = [
    "Axis #1",
    "Axis #2",
    "Axis #3",
    "Axis #4",
    "Axis #5",
    "Axis #6",
    "Axis #7",
    "Axis #8"
]

for axis in usable_axes:
    non_zero = (analysis_data[axis] > 0).sum()
    zero = (analysis_data[axis] == 0).sum()

    print(f"{axis}:")
    print(f"  Zero values: {zero}")
    print(f"  Non-zero values: {non_zero}")

Axis #1:
  Zero values: 25914
  Non-zero values: 13758
Axis #2:
  Zero values: 25822
  Non-zero values: 13850
Axis #3:
  Zero values: 25844
  Non-zero values: 13828
Axis #4:
  Zero values: 26004
  Non-zero values: 13668
Axis #5:
  Zero values: 25864
  Non-zero values: 13808
Axis #6:
  Zero values: 26073
  Non-zero values: 13599
Axis #7:
  Zero values: 26010
  Non-zero values: 13662
Axis #8:
  Zero values: 25958
  Non-zero values: 13714


### Test 3: Descriptive Statistics

We now examine the descriptive statistics for the eight usable robot axes.

The statistics show the typical and extreme values for each axis, including the mean, standard deviation, minimum, maximum, and percentiles.

Because each axis can have a different measurement range, these statistics help us understand the normal behaviour of each axis before identifying potential anomalies.


In [4]:
print(
    analysis_data[usable_axes].describe()
)

            Axis #1       Axis #2       Axis #3       Axis #4       Axis #5  \
count  39672.000000  39672.000000  39672.000000  39672.000000  39672.000000   
mean       0.725743      3.613374      2.710336      0.620222      0.954521   
std        2.162120      6.879962      5.111901      1.574897      2.100186   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        0.000000      0.000000      0.000000      0.000000      0.000000   
50%        0.000000      0.000000      0.000000      0.000000      0.000000   
75%        0.312710      4.217190      4.586190      0.516190      0.800090   
max       23.609300     51.713230     41.855560     15.666300     20.750760   

            Axis #6       Axis #7       Axis #8  
count  39672.000000  39672.000000  39672.000000  
mean       0.599427      0.870145      0.102214  
std        1.815498      2.166811      0.423075  
min        0.000000      0.000000      0.000000  
25%        0.000000      0.000000     

### Test 4: Identify Potential Anomalies

We now identify potential anomalous measurements for each usable robot axis.

A measurement is considered a potential anomaly when it is higher than the 99th-percentile value for that individual axis.

The 99th percentile is used because the dataset does not provide confirmed failure labels. Therefore, this analysis identifies unusually high measurements rather than confirming equipment failure.

Each axis uses its own threshold because the axes have different measurement ranges.


In [5]:
analysis_data["Anomaly"] = False

for axis in usable_axes:

    threshold = analysis_data[axis].quantile(0.99)

    print(f"{axis} threshold: {threshold:.4f}")

    analysis_data.loc[
        analysis_data[axis] > threshold,
        "Anomaly"
    ] = True

print("\nTotal records:", len(analysis_data))
print("Potential anomalies:", analysis_data["Anomaly"].sum())
print("Normal records:", (~analysis_data["Anomaly"]).sum())

Axis #1 threshold: 11.3616
Axis #2 threshold: 31.3653
Axis #3 threshold: 24.7233
Axis #4 threshold: 8.3439
Axis #5 threshold: 9.8850
Axis #6 threshold: 10.3496
Axis #7 threshold: 8.0966
Axis #8 threshold: 2.9683

Total records: 39672
Potential anomalies: 2591
Normal records: 37081


### Anomaly Findings

The analysis identified **2,591 readings containing at least one potential anomalous measurement** out of **39,672 total readings**.

These readings represent approximately **6.5% of the dataset**.

A potential anomaly means that at least one axis produced a measurement above its individual 99th-percentile threshold. This indicates an unusually high reading compared with the observed values for that axis.

The remaining **37,081 readings** did not contain an anomalous axis.

These results identify unusual measurements for further investigation and do not confirm that the robot has failed.


### Test 5: Identify Potential Maintenance Alerts

A reading with one anomalous axis may be an isolated unusual measurement.

When two or more axes show anomalous measurements in the same reading, this provides a stronger indication of unusual robot behaviour.

Therefore, we count the number of anomalous axes in each reading and use readings with **2 or more anomalous axes** as potential Maintenance Notification alerts.

These alerts are warnings for further inspection and do not confirm that a robot component has failed.


In [6]:
analysis_data["Anomaly_Axis_Count"] = 0

for axis in usable_axes:

    threshold = analysis_data[axis].quantile(0.99)

    analysis_data["Anomaly_Axis_Count"] += (
        analysis_data[axis] > threshold
    ).astype(int)

print(
    analysis_data["Anomaly_Axis_Count"]
    .value_counts()
    .sort_index()
)

multi_axis_events = analysis_data[
    analysis_data["Anomaly_Axis_Count"] >= 2
].copy()

print(
    "\nPotential maintenance-alert events:",
    len(multi_axis_events)
)

Anomaly_Axis_Count
0    37081
1     2249
2      314
3       28
Name: count, dtype: int64

Potential maintenance-alert events: 342


### Maintenance Notification Alerts

The analysis identified the following robot readings based on the number of anomalous axes:

* **37,081 readings:** No anomalous axes
* **2,249 readings:** 1 anomalous axis
* **314 readings:** 2 anomalous axes
* **28 readings:** 3 anomalous axes

Readings with **2 or more anomalous axes** are treated as potential Maintenance Notification events.

This resulted in **342 potential maintenance-alert events**.

These alerts indicate unusual robot behaviour and should be reviewed by a maintenance team. They are **warnings for further inspection and do not confirm equipment failure**.


### Test 6: View Potential Maintenance Alerts

The following cell displays examples of readings that triggered a potential Maintenance Notification.

These readings contain two or more anomalous axes in the same measurement. Reviewing these records helps show when unusual robot behaviour occurred and which axis measurements contributed to the alert.


In [7]:
display(
    multi_axis_events[
        ["Time"] + usable_axes + ["Anomaly_Axis_Count"]
    ].head(10)
)

,Time,Axis #1,Axis #2,Axis #3,Axis #4,Axis #5,Axis #6,Axis #7,Axis #8,Anomaly_Axis_Count
130,2022-10-17T12:22:45.266Z,12.01312,9.01423,6.80021,12.18204,5.80711,0.61943,0.46504,0.09810,2
353,2022-10-17T12:36:56.453Z,20.76889,12.22984,10.96468,1.57437,14.04031,1.10980,0.04770,0.09810,2
396,2022-10-17T12:38:22.597Z,11.67436,3.21560,26.56827,0.82590,3.27779,2.78742,0.54851,1.19682,2
627,2022-10-17T12:46:07.684Z,2.44953,0.63258,33.15762,2.83903,5.31674,11.35614,0.07155,0.13734,2
1129,2022-10-17T13:08:32.404Z,1.38112,43.33158,6.90564,1.36790,19.33124,0.20648,0.07155,0.11772,2
1308,2022-10-17T13:14:32.081Z,1.95441,44.43859,6.16763,0.33552,17.88591,0.30971,0.05962,0.03924,2
1506,2022-10-17T13:21:14.736Z,15.79165,44.22773,7.06379,10.78833,4.23274,0.05162,0.02385,0.21582,3
1627,2022-10-17T13:25:14.350Z,9.85024,34.15920,14.39114,11.04642,1.21304,0.77428,0.01192,0.03924,2
1652,2022-10-17T13:26:05.574Z,0.44300,40.37955,6.53664,1.44533,17.86010,0.02581,0.04770,0.09810,2
1676,2022-10-17T13:26:55.255Z,15.86983,45.12388,7.27464,10.86576,4.18112,0.05162,0.01192,0.07848,3


### Robot State Summary

Based on the analysis, most robot readings fall within the expected measurement range, while a smaller number contain unusually high measurements.

The analysis identified **2,591 readings with at least one potential anomaly** and **342 readings with anomalies across two or more axes**.

The 342 multi-axis events are treated as potential Maintenance Notification alerts because simultaneous unusual measurements may indicate a change in robot behaviour.

These results should be used to support maintenance inspection rather than to confirm equipment failure. Further investigation and additional sensor or maintenance information would be required to determine the actual condition of the robot.


### Talking Point: Robot Monitoring

The analysis shows that most robot readings remain within the observed measurement range, while a smaller number contain unusually high values.

Using the 99th-percentile thresholds, we identified **2,591 potential anomalous readings**. Among these, **342 readings contained unusual measurements across two or more axes** and were treated as potential Maintenance Notification events.

This demonstrates how robot sensor data can be used to monitor robot behaviour and identify readings that may require further inspection.
